# Let's Build a Byte-Level BPE Tokenizer

This is a small, hands-on Python project for understanding byte-level Byte Pair Encoding (BPE). We will turn text into UTF-8 bytes, teach the tokenizer which byte pairs are common, and then use those learned rules to encode and decode text.

The learning approach is inspired by Andrej Karpathy's [BPE tokenizer tutorial](https://www.youtube.com/watch?v=zduSFxRajkE). The friendlier walkthrough, multilingual examples, and validation checks are additions to this project. This is a learning implementation, not a production tokenizer.

## 1. Start with text: turn it into UTF-8 bytes

Computers store text as numbers. Here, we use UTF-8, which converts every character into one or more byte values between 0 and 255. Starting with all 256 possible bytes means this tokenizer can still represent characters it has never seen during training.

In [23]:
training_text = """Hello world!

안녕하세요 세계

こんにちは世界

你好，世界！

नमस्ते दुनिया

ನಮಸ್ಕಾರ ಜಗತ್ತು

مرحبا بالعالم

Привет мир

Γειά σου κόσμε

สวัสดีชาวโลก"""

# UTF-8 produces integers in the inclusive range 0..255.
byte_ids = list(training_text.encode("utf-8"))

print(training_text)
print(f"Characters: {len(training_text)}")
print(f"UTF-8 bytes: {len(byte_ids)}")
print(f"First 30 byte IDs: {byte_ids[:30]}")

Hello world!

안녕하세요 세계

こんにちは世界

你好，世界！

नमस्ते दुनिया

ನಮಸ್ಕಾರ ಜಗತ್ತು

مرحبا بالعالم

Привет мир

Γειά σου κόσμε

สวัสดีชาวโลก
Characters: 127
UTF-8 bytes: 274
First 30 byte IDs: [72, 101, 108, 108, 111, 32, 119, 111, 114, 108, 100, 33, 10, 10, 236, 149, 136, 235, 133, 149, 237, 149, 152, 236, 132, 184, 236, 154, 148, 32]


## 2. Look for byte pairs that often appear together

BPE learns by looking at neighbours. At first, every token is just one byte. We count each pair of adjacent tokens and find the pair that appears most often. Later, a single token can stand for a longer sequence of bytes.

In [24]:
def get_stats(ids):
    """Return frequencies for every adjacent pair in a token sequence."""
    counts = {}
    for pair in zip(ids, ids[1:]):
        counts[pair] = counts.get(pair, 0) + 1
    return counts

pair_counts = get_stats(byte_ids)
top_pair = max(pair_counts, key=pair_counts.get)
print(f"Most frequent initial pair: {top_pair}")
print(f"Occurrences: {pair_counts[top_pair]}")

Most frequent initial pair: (224, 184)
Occurrences: 11


## 3. Give a frequent pair its own token

Once we choose a frequent pair, we replace every non-overlapping occurrence with one new token ID. IDs 0–255 remain the original individual bytes. New, learned tokens begin at ID 256.

In [25]:
def merge(ids, pair, new_token_id):
    """Replace every non-overlapping occurrence of *pair* in *ids*."""
    merged = []
    position = 0
    while position < len(ids):
        if position < len(ids) - 1 and (ids[position], ids[position + 1]) == pair:
            merged.append(new_token_id)
            position += 2
        else:
            merged.append(ids[position])
            position += 1
    return merged

one_merge = merge(byte_ids, top_pair, 256)
print(f"Bytes before one merge: {len(byte_ids)}")
print(f"Tokens after one merge: {len(one_merge)}")

Bytes before one merge: 274
Tokens after one merge: 263


## 4. Train the tokenizer, one merge at a time

Now we repeat the same simple loop: count pairs, choose the most common one, and merge it. Every chosen pair is saved with its new token ID. Those saved merge rules are what the tokenizer learns from the training text.

In [26]:
def train_bpe(text, vocab_size=300):
    """Learn BPE merges from UTF-8 text and return the merge-rule dictionary."""
    if vocab_size < 256:
        raise ValueError("vocab_size must be at least 256 for the byte vocabulary")

    ids = list(text.encode("utf-8"))
    merges = {}

    for merge_number in range(vocab_size - 256):
        stats = get_stats(ids)
        if not stats:
            break  # Fewer than two tokens remain.

        pair = max(stats, key=stats.get)
        new_token_id = 256 + merge_number
        ids = merge(ids, pair, new_token_id)
        merges[pair] = new_token_id

    return merges


merges = train_bpe(training_text, vocab_size=300)
print(f"Learned merges: {len(merges)}")
print(f"Vocabulary size: {256 + len(merges)}")
print(f"First five rules: {list(merges.items())[:5]}")

Learned merges: 44
Vocabulary size: 300
First five rules: [((224, 184), 256), ((224, 178), 257), ((10, 10), 258), ((224, 164), 259), ((227, 129), 260)]


## 5. Turn token IDs back into readable text

To decode, we need to know what bytes each token represents. Base tokens map to one byte, while learned tokens map to the bytes of the pair that created them. We join the bytes and decode them as UTF-8. Using strict UTF-8 helps us notice invalid token sequences instead of silently changing them.

In [27]:
def build_vocab(merges):
    """Build token ID -> byte sequence mappings from learned merge rules."""
    vocab = {token_id: bytes([token_id]) for token_id in range(256)}
    for (left, right), token_id in merges.items():
        vocab[token_id] = vocab[left] + vocab[right]
    return vocab


vocab = build_vocab(merges)


def decode(ids):
    """Convert token IDs back to text; reject token IDs outside this vocabulary."""
    unknown_ids = [token_id for token_id in ids if token_id not in vocab]
    if unknown_ids:
        raise ValueError(f"Unknown token ID(s): {unknown_ids}")
    return b"".join(vocab[token_id] for token_id in ids).decode("utf-8", errors="strict")

## 6. Encode new text using what the tokenizer learned

To encode new text, we again begin with UTF-8 bytes. Then we apply only the merge rules learned during training. When several learned pairs are available, the earliest learned rule wins. We do not learn new rules from the input being encoded. Because all 256 byte values are present from the start, even bytes not seen during training are still handled.

In [28]:
def encode(text):
    """Convert text to token IDs using the learned BPE merge rules."""
    ids = list(text.encode("utf-8"))

    while len(ids) >= 2:
        stats = get_stats(ids)
        learned_pairs = [pair for pair in stats if pair in merges]
        if not learned_pairs:
            break

        pair = min(learned_pairs, key=merges.__getitem__)
        ids = merge(ids, pair, merges[pair])

    return ids


example = "Hello world!"
example_ids = encode(example)
print(f"Text: {example!r}")
print(f"Token IDs: {example_ids}")
print(f"Decoded: {decode(example_ids)!r}")

Text: 'Hello world!'
Token IDs: [298]
Decoded: 'Hello world!'


## 7. Check that no text is lost along the way

The key check is a round trip: after encoding and then decoding, we should get back exactly the text we started with. The tests below use several writing systems and also include text that was not in the training data. This shows why a byte-level tokenizer does not need an unknown-byte token.

In [29]:
test_cases = {
    "empty": "",
    "English": "Hello, tokenizer!",
    "Kannada": "ನಮಸ್ಕಾರ ಜಗತ್ತು",
    "Hindi": "नमस्ते दुनिया",
    "Japanese": "こんにちは世界",
    "Chinese": "你好，世界！",
    "Korean": "안녕하세요 세계",
    "Arabic": "مرحبا بالعالم",
    "Russian": "Привет мир",
    "Unseen UTF-8 input": "မြန်မာစာ — 🧪",
}

for language, sample in test_cases.items():
    encoded = encode(sample)
    decoded = decode(encoded)
    assert decoded == sample, f"Round trip failed for {language}"
    print(f"PASS: {language:18} {len(encoded):3} token(s)")

# Invalid token IDs are reported clearly instead of being treated as bytes.
try:
    decode([9999])
except ValueError as error:
    print(f"PASS: invalid token ID check ({error})")

PASS: empty                0 token(s)
PASS: English             13 token(s)
PASS: Kannada             23 token(s)
PASS: Hindi               21 token(s)
PASS: Japanese            12 token(s)
PASS: Chinese             11 token(s)
PASS: Korean              18 token(s)
PASS: Arabic              14 token(s)
PASS: Russian             15 token(s)
PASS: Unseen UTF-8 input  33 token(s)
PASS: invalid token ID check (Unknown token ID(s): [9999])


## Where could this go next?

This notebook intentionally keeps the implementation small so the core idea stays easy to follow. A production tokenizer would also need a much larger corpus, faster pair counting, special tokens, a way to save and load its vocabulary, and carefully defined text-normalization and pre-tokenization rules.